In [90]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, LabelEncoder, FunctionTransformer, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.base import BaseEstimator, TransformerMixin

from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import joblib
import warnings
warnings.filterwarnings('ignore')

In [91]:
# Load the Dataset
path = "California_House_Prices_Data.csv"
df = pd.read_csv(path)
df

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,ocean_proximity,median_house_value
0,-122.23,37.88,41,880,129.0,322,126,8.3252,NEAR BAY,452600
1,-122.22,37.86,21,7099,1106.0,2401,1138,8.3014,NEAR BAY,358500
2,-122.24,37.85,52,1467,190.0,496,177,7.2574,NEAR BAY,352100
3,-122.25,37.85,52,1274,235.0,558,219,5.6431,NEAR BAY,341300
4,-122.25,37.85,52,1627,280.0,565,259,3.8462,NEAR BAY,342200
...,...,...,...,...,...,...,...,...,...,...
20635,-121.09,39.48,25,1665,374.0,845,330,1.5603,INLAND,78100
20636,-121.21,39.49,18,697,150.0,356,114,2.5568,INLAND,77100
20637,-121.22,39.43,17,2254,485.0,1007,433,1.7000,INLAND,92300
20638,-121.32,39.43,18,1860,409.0,741,349,1.8672,INLAND,84700


In [92]:
# Custom Transformer: Outlier Capper
class OutlierCapper(BaseEstimator, TransformerMixin):
    def __init__(self, factor=2.0):
        self.factor = factor
        self.bounds = {}

    def fit(self, X, y=None):
        X = pd.DataFrame(X).copy()
        for col in X.columns:
            if np.issubdtype(X[col].dtype, np.number):
                Q1, Q3 = np.percentile(X[col].dropna(), [25, 75])
                IQR = Q3 - Q1
                lower = Q1 - self.factor * IQR
                upper = Q3 + self.factor * IQR
                self.bounds[col] = (lower, upper)
        return self

    def transform(self, X):
        X = pd.DataFrame(X).copy()
        for col, (lower, upper) in self.bounds.items():
            X[col] = np.clip(X[col], lower, upper)
        return X

In [93]:
# Feature Engineering Functions
def add_features(df):
    df = df.copy()
    df["rooms_per_household"] = df["total_rooms"] / df["households"]
    df["bedrooms_per_room"] = df["total_bedrooms"] / df["total_rooms"]
    df["population_per_household"] = df["population"] / df["households"]
    df["rooms_per_person"] = df["total_rooms"] / df["population"]
    df["population_per_room"] = df["population"] / df["total_rooms"]
    df["households_per_population"] = df["households"] / df["population"]
    df["bedrooms_per_household"] = df["total_bedrooms"] / df["households"]
    df["income_x_rooms_per_household"] = df["median_income"] * df["rooms_per_household"]

    # log transforms
    for col in ["total_rooms", "total_bedrooms", "population", "median_income"]:
        df[f"log_{col}"] = np.log1p(df[col])

    # binning
    df["age_bin"] = pd.cut(df["housing_median_age"], bins=[0, 10, 20, 30, 40, 50], labels=False)
    df["income_bin"] = pd.cut(df["median_income"], bins=5, labels=False)
    df["lat_bin"] = pd.cut(df["latitude"], bins=5, labels=False)
    df["lon_bin"] = pd.cut(df["longitude"], bins=5, labels=False)

    return df

feature_engineering = FunctionTransformer(add_features)


In [94]:
# Define Columns
categorical = ["ocean_proximity"]
numeric = [
    "longitude", "latitude", "housing_median_age", "total_rooms", "total_bedrooms",
    "population", "households", "median_income",
    # engineered columns will also be handled automatically
]


In [95]:

# Preprocessor
preprocessor = ColumnTransformer([
    ("num", Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("outlier_cap", OutlierCapper(factor=1.5)),
        ("scaler", StandardScaler())
    ]), numeric),
    ("cat", Pipeline([
        ("encoder", OneHotEncoder(handle_unknown="ignore"))
    ]), categorical)
], remainder='passthrough')

In [96]:
# Final Pipeline
model = Pipeline([
    ("features", feature_engineering),
    ("preprocessor", preprocessor),
('xgb', XGBRegressor(
    n_estimators=800,        
    learning_rate=0.05,      
    max_depth=6,             
    subsample=0.8,           
    colsample_bytree=0.8,    
    gamma=1,                 
    random_state=42,
    n_jobs=-1
))])

In [97]:
# Train the pipeline
X = df.drop("median_house_value", axis=1)
y = df["median_house_value"]


In [98]:
# Encode 'ocean_proximity' using LabelEncoder
le = LabelEncoder()
X['ocean_proximity'] = le.fit_transform(X['ocean_proximity'])


In [99]:
# Train & Evaluate
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model.fit(X_train, y_train)
r2 = model.score(X_test, y_test)
y_pred = model.predict(X_test)

cv_scores = cross_val_score(model, X, y, cv=5, scoring="r2")
mae = mean_absolute_error(y_test, y_pred)       # Average absolute error
mse = mean_squared_error(y_test, y_pred)        # Mean squared error
rmse = np.sqrt(mse)                             # Root Mean Squared Error

# Display All Metrics
print("✅ Model Evaluation Metrics:")
print(f"Test R² Score: {r2:.4f}")
print(f"Mean CV R² Score: {cv_scores.mean():.4f}")
print(f"CV R² Scores: {cv_scores}")
print(f"Mean Absolute Error (MAE): {mae:.2f}")
print(f"Mean Squared Error (MSE): {mse:.2f}")
print(f"Root Mean Squared Error (RMSE): {rmse:.2f}")

✅ Model Evaluation Metrics:
Test R² Score: 0.8462
Mean CV R² Score: 0.6657
CV R² Scores: [0.62398839 0.65912247 0.74801975 0.57785732 0.71936727]
Mean Absolute Error (MAE): 29108.42
Mean Squared Error (MSE): 2014920832.00
Root Mean Squared Error (RMSE): 44887.87


In [100]:
sample = {
    'longitude': -122.23,
    'latitude': 37.88,
    'housing_median_age': 41,
    'total_rooms': 880,
    'total_bedrooms': 129,
    'population': 322,
    'households': 126,
    'median_income': 8.3252,
    'ocean_proximity': 'NEAR BAY'
}

input_df = pd.DataFrame([sample])
print("Sample Prediction:", round(model.predict(input_df)[0], 2))

Sample Prediction: 427815.2


In [101]:
import dill
import joblib

model = joblib.load("california_house_model.pkl")

# Save using dill
with open("california_house_model_dill.pkl", "wb") as f:
    dill.dump(model, f)